# Efficient Global Optimization · Interactive Dashboard

**Fixed vs. adaptive Gaussian-process hyperparameters** on the modified Branin benchmark. Choose settings, then click **Run comparison** to update the figures and numerical results.

In [ ]:
from pathlib import Path
import sys

# Support JupyterLab started at the repository root or in the notebook directory,
# as well as Voilà/Binder launching from either location.
def find_repository_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "global_optimization" / "ego" / "ego.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find repository root. Launch this notebook from the cloned "
        "Numerical-Optimization-and-Iterative-Methods repository."
    )

repository_root = find_repository_root()
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

from global_optimization.ego import (
    efficient_global_optimization,
    modified_branin_normalized,
)


def objective(x):
    return float(modified_branin_normalized(x[0], x[1]))

In [ ]:
# Interactive controls
control_layout = widgets.Layout(width="235px")
control_style = {"description_width": "125px"}

initial_widget = widgets.IntSlider(
    value=9, min=4, max=15, step=1, description="Initial samples",
    continuous_update=False, style=control_style, layout=control_layout,
)
budget_widget = widgets.IntSlider(
    value=30, min=16, max=50, step=1, description="Total budget",
    continuous_update=False, style=control_style, layout=control_layout,
)
theta_widget = widgets.FloatSlider(
    value=0.2, min=0.05, max=1.0, step=0.05, description="Length scale θ",
    readout_format=".2f", continuous_update=False,
    style=control_style, layout=control_layout,
)
xi_widget = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05, description="Exploration ξ",
    readout_format=".2f", continuous_update=False,
    style=control_style, layout=control_layout,
)
seed_widget = widgets.BoundedIntText(
    value=42, min=0, max=999999, description="Random seed",
    style=control_style, layout=control_layout,
)
run_button = widgets.Button(
    description="Run comparison", button_style="success", icon="play",
    layout=widgets.Layout(width="205px", height="38px"),
)
status = widgets.HTML(
    '<span style="color:#7a5b00">Ready. Click <b>Run comparison</b> to begin.</span>'
)
plot_output = widgets.Output()
table_output = widgets.Output()
results_accordion = widgets.Accordion(children=[table_output])
results_accordion.set_title(0, "Numerical results")
results_accordion.selected_index = None
results_accordion.layout.display = "none"

controls = widgets.VBox([
    widgets.HBox([initial_widget, budget_widget]),
    widgets.HBox([theta_widget, xi_widget]),
    seed_widget,
    run_button,
    status,
])


def mark_stale(change):
    status.value = (
        '<span style="color:#9a6700">⚠ Parameters changed. '
        'Click <b>Run comparison</b> to update the results.</span>'
    )


for control in (initial_widget, budget_widget, theta_widget, xi_widget, seed_widget):
    control.observe(mark_stale, names="value")


def run_comparison(_):
    if initial_widget.value > budget_widget.value:
        status.value = '<span style="color:#b42318">Initial samples exceed total budget.</span>'
        return

    run_button.disabled = True
    status.value = '<span style="color:#3b5b81">Running Fixed and Adaptive EGO…</span>'
    with plot_output:
        plot_output.clear_output(wait=True)
    with table_output:
        table_output.clear_output(wait=True)

    try:
        settings = dict(
            objective=objective,
            bounds=[(0.0, 1.0), (0.0, 1.0)],
            n_initial=initial_widget.value,
            max_evaluations=budget_widget.value,
            theta=theta_widget.value,
            sigma=1.0,
            nugget=1e-8,
            xi=xi_widget.value,
            n_restarts=10,
            random_state=seed_widget.value,
        )
        fixed = efficient_global_optimization(**settings, estimate_parameters=False)
        adaptive = efficient_global_optimization(**settings, estimate_parameters=True)

        evaluations = np.arange(1, settings["max_evaluations"] + 1)
        fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), constrained_layout=True)
        ax = axes[0]
        ax.plot(evaluations, fixed.best_history, lw=2.1, label="Fixed EGO", color="#2563eb")
        ax.plot(evaluations, adaptive.best_history, lw=2.1, label="Adaptive EGO", color="#db5b36")
        ax.axvline(settings["n_initial"], ls="--", lw=1.1, color="#6b7280", label="Initial design ends")
        ax.set(xlabel="Objective evaluations", ylabel="Best objective so far", title="Convergence")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False, fontsize=9)

        ax = axes[1]
        ax.scatter(fixed.X[:settings["n_initial"], 0], fixed.X[:settings["n_initial"], 1],
                   marker="o", facecolors="none", edgecolors="#64748b", s=52, label="Shared initial design")
        ax.scatter(fixed.X[settings["n_initial"]:, 0], fixed.X[settings["n_initial"]:, 1],
                   marker=".", s=42, color="#2563eb", label="Fixed EGO")
        ax.scatter(adaptive.X[settings["n_initial"]:, 0], adaptive.X[settings["n_initial"]:, 1],
                   marker="x", s=35, color="#db5b36", label="Adaptive EGO")
        ax.scatter(*fixed.x_best, s=95, marker="*", color="#1d4ed8", zorder=4)
        ax.scatter(*adaptive.x_best, s=95, marker="*", color="#c2410c", zorder=4)
        ax.set(xlabel="Normalized y₁", ylabel="Normalized y₂", title="Evaluated locations",
               xlim=(-0.03, 1.03), ylim=(-0.03, 1.03))
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
        ax.legend(
            frameon=False,
            fontsize=8,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0,
        )

        with plot_output:
            display(fig)
        plt.close(fig)

        rows = []
        for name, result in (("Fixed EGO", fixed), ("Adaptive EGO", adaptive)):
            rows.append({
                "Method": name,
                "Initial best": result.best_history[settings["n_initial"] - 1],
                "Final best": result.f_best,
                "Best y₁": result.x_best[0],
                "Best y₂": result.x_best[1],
                "Evaluations": result.n_evaluations,
            })
        frame = pd.DataFrame(rows).set_index("Method")
        with table_output:
            display(frame.style.format({
                "Initial best": "{:.6f}", "Final best": "{:.6f}",
                "Best y₁": "{:.6f}", "Best y₂": "{:.6f}",
                "Evaluations": "{:d}",
            }))
        results_accordion.layout.display = ""
        results_accordion.selected_index = None
        status.value = (
            '<span style="color:#137333">✓ Comparison complete — '
            f'Fixed: <b>{fixed.f_best:.6f}</b>; '
            f'Adaptive: <b>{adaptive.f_best:.6f}</b>. '
            'Expand <b>Numerical results</b> for details.</span>'
        )
    except Exception as exc:
        status.value = f'<span style="color:#b42318">Error: {type(exc).__name__}: {exc}</span>'
        raise
    finally:
        run_button.disabled = False


run_button.on_click(run_comparison)
display(widgets.VBox([controls, plot_output, results_accordion]))